In [17]:
import pandas as pd
from scipy.sparse import coo_matrix
import matplotlib.pyplot as plt
from graph import build_pseudo_real_graph, build_real_spot_graph, build_spot_gene_graph
from process import random_mix_with_dominant
from sklearn.preprocessing import LabelEncoder,MinMaxScaler

import scanpy as sc
import anndata as ad
import numpy as np

In [18]:

# spot * gene
expr_real = pd.read_csv('./MOB/spatial_count.csv',index_col=0)

adata = ad.AnnData(expr_real)
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
sc.pp.highly_variable_genes(adata, flavor='seurat', n_top_genes=2000)
adata = adata[:, adata.var['highly_variable']]
expr_real = pd.DataFrame(
    adata.X.toarray() if not isinstance(adata.X, np.ndarray) else adata.X,
    index=adata.obs_names,
    columns=adata.var_names
)

coords_real = pd.read_csv('.\MOB\spatial_location.csv',index_col=0)
coords_real.columns = ['x','y']
# coords_real = coords_real.drop('',axis=1)


# gene * cell
expr_sc = pd.read_csv('./MOB/sc_count.txt',sep='\t')
expr_sc = expr_sc.set_index('cell_id')
expr_sc = expr_sc.T # cell * gene


sc_labels = pd.read_csv('./MOB/sc_labels.txt',header=None)
sc_labels = sc_labels[0].values  # 获取为一维数组


adata = ad.AnnData(expr_sc)
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
sc.pp.highly_variable_genes(adata, flavor='seurat', n_top_genes=2000)
adata = adata[:, adata.var['highly_variable']]

expr_sc = pd.DataFrame(
    adata.X.toarray() if not isinstance(adata.X, np.ndarray) else adata.X,
    index=adata.obs_names,
    columns=adata.var_names
)

C:\Users\lenovo\AppData\Local\Temp\ipykernel_15308\780262213.py:5: FutureWarning: X.dtype being converted to np.float32 from int64. In the next version of anndata (0.9) conversion will not be automatic. Pass dtype explicitly to avoid this warning. Pass `AnnData(X, dtype=X.dtype, ...)` to get the future behavour.
  adata = ad.AnnData(expr_real)
C:\Users\lenovo\AppData\Local\Temp\ipykernel_15308\780262213.py:31: FutureWarning: X.dtype being converted to np.float32 from int64. In the next version of anndata (0.9) conversion will not be automatic. Pass dtype explicitly to avoid this warning. Pass `AnnData(X, dtype=X.dtype, ...)` to get the future behavour.
  adata = ad.AnnData(expr_sc)


In [19]:
expr_sc,expr_real,coords_real

(cell_id                       Olig1    Nfasc   Tmem141       Sln     Ugt8a  \
 X10X49_4_ATACCTGGACAG.     0.000000  0.00000  0.000000  1.882199  0.000000   
 X10X28_5_AAATTCGAAGATGA.1  0.000000  0.00000  0.000000  0.000000  0.000000   
 X10X28_5_GAACTGTGGACGAG.1  0.000000  0.00000  0.000000  0.000000  0.000000   
 X10X49_4_CTAGCTGCGGAA.     0.000000  0.00000  0.000000  0.000000  0.000000   
 X10X49_4_AACTCTTGGTCA.     0.000000  0.00000  0.000000  0.000000  0.000000   
 ...                             ...      ...       ...       ...       ...   
 X10X49_4_AGGTCTTAGCCA.     0.000000  0.00000  0.000000  0.000000  1.666167   
 X10X04_4_AAACCGTGCTCGCT.1  3.033972  1.34173  2.248865  0.000000  2.716566   
 X10X28_4_ATGTCGGAACACCA.1  0.000000  0.00000  0.000000  0.000000  0.000000   
 X10X49_4_CATGGACTCCAC.     0.000000  0.00000  1.603068  0.000000  0.000000   
 X10X49_5_ACTGACGGTGTT.     0.000000  0.00000  0.000000  0.000000  0.000000   
 
 cell_id                         Mag       Mal    

In [20]:

le = LabelEncoder()
ys = le.fit_transform(sc_labels)  # 例如 'iNeuron' -> 0, 'Astrocyte' -> 1, ...
print("类别数:", len(le.classes_))
print("类别名:", le.classes_)

Xs = expr_sc.values


pseudo_X, pseudo_y = random_mix_with_dominant(
    Xs, ys,
    nmix=7,
    n_samples=500,      
    n_dominant=80,
    dominant_ratio=0.7,
    seed=42
)



pseudo_X_df = pd.DataFrame(pseudo_X, columns=expr_sc.columns)
pseudo_y_df = pd.DataFrame(pseudo_y, columns=le.classes_)  


adata_pseudo = ad.AnnData(pseudo_X_df)
sc.pp.normalize_total(adata_pseudo, target_sum=1e4)
sc.pp.log1p(adata_pseudo)


pseudo_X_df = pd.DataFrame(
    adata_pseudo.X.toarray() if not isinstance(adata_pseudo.X, np.ndarray) else adata_pseudo.X,
    index=adata_pseudo.obs_names,
    columns=adata_pseudo.var_names
)

pseudo_X_df.to_csv('./MOB/500/pseudo_spot_expression.csv', index=False)
pseudo_y_df.to_csv('./MOB/500/pseudo_spot_label_fractions.csv', index=False)

类别数: 5
类别名: ['Astrocytes' 'Immune' 'Neurons' 'Oligos' 'Vascular']


C:\Users\lenovo\AppData\Local\Temp\ipykernel_15308\1466547624.py:25: FutureWarning: X.dtype being converted to np.float32 from float64. In the next version of anndata (0.9) conversion will not be automatic. Pass dtype explicitly to avoid this warning. Pass `AnnData(X, dtype=X.dtype, ...)` to get the future behavour.
  adata_pseudo = ad.AnnData(pseudo_X_df)
D:\Anaconda\Anaconda\envs\STAMapper\lib\site-packages\anndata\_core\anndata.py:121: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


In [21]:
expr_pseudo = pseudo_X_df

In [22]:

adj_real_real = build_real_spot_graph(expr_real, coords_real, k=10)


shared_genes = expr_real.columns.intersection(expr_pseudo.columns)

expr_real = expr_real[shared_genes].copy()
expr_pseudo = expr_pseudo[shared_genes].copy()
print(f"共同基因数量: {len(shared_genes)}")


expr_real.to_csv('./MOB/500/real_spot_shared_genes.csv', index=True)


expr_pseudo.to_csv('./MOB/500/pseudo_spot_shared_genes.csv', index=True)

adj_real_pseudo = build_pseudo_real_graph(expr_real, expr_pseudo, k=40)



adj_spot_gene = build_spot_gene_graph(expr_real,expr_pseudo, threshold=0.005)

共同基因数量: 1150


In [23]:
adj_real_real,adj_real_pseudo,adj_spot_gene

(<260x260 sparse matrix of type '<class 'numpy.float64'>'
 	with 2018 stored elements in COOrdinate format>,
 <760x760 sparse matrix of type '<class 'numpy.float32'>'
 	with 14232 stored elements in COOrdinate format>,
 <760x1150 sparse matrix of type '<class 'numpy.float32'>'
 	with 148736 stored elements in COOrdinate format>)

In [24]:

adj_dense = adj_real_real.toarray()

cell_ids = expr_real.index.tolist()

adj_df_named = pd.DataFrame(adj_dense, index=cell_ids, columns=cell_ids)

adj_df_named.to_csv('./MOB/500/adj_real_real.csv')


n_real = expr_real.shape[0]
n_pseudo = expr_pseudo.shape[0]
adj_dense = adj_real_pseudo.toarray()

real_ids = [f"real_{i}" for i in expr_real.index]
pseudo_ids = [f"pseudo_{i}" for i in expr_pseudo.index]
node_ids = real_ids + pseudo_ids

adj_df = pd.DataFrame(adj_dense, index=node_ids, columns=node_ids)

adj_df = adj_df.iloc[:n_real, n_real:]
adj_df.to_csv("./MOB/500/adj_real_pseudo.csv")





adj_matrix = pd.DataFrame(
    adj_spot_gene.toarray(),
    index=list(expr_real.index) + list(expr_pseudo.index),
    columns=expr_real.columns  
)
adj_matrix_real = adj_matrix.iloc[:n_real]
adj_matrix_pseudo = adj_matrix.iloc[n_real:]

adj_matrix_real.to_csv("./MOB/500/adj_realspot_gene.csv")
adj_matrix_pseudo.to_csv("./MOB/500/adj_pseuspot_gene.csv")
